# الفصل 7: بناء تطبيقات الدردشة  
## بداية سريعة مع واجهة برمجة التطبيقات OpenAI  

تم تكييف هذه الدفتر من [مستودع عينات Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) الذي يتضمن دفاتر تتصل بخدمات [Azure OpenAI](aoai-assignment.ipynb).  

تعمل واجهة برمجة التطبيقات OpenAI بلغة بايثون مع نماذج Azure OpenAI أيضًا، مع بعض التعديلات. تعرّف أكثر على الفروقات هنا: [كيفية التبديل بين نقاط نهاية OpenAI و Azure OpenAI باستخدام بايثون](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)  


# نظرة عامة  
"نماذج اللغة الكبيرة هي دوال تحوّل النص إلى نص. بالنظر إلى سلسلة نصية إدخال، يحاول نموذج اللغة الكبير التنبؤ بالنص الذي سيأتي بعده"(1). ستقدم هذه الدفترية "الانطلاقة السريعة" للمستخدمين مفاهيم عالية المستوى حول نماذج اللغة الكبيرة، والمتطلبات الأساسية للحزمة للبدء مع AML، ومقدمة مرنة لتصميم المطالبات، والعديد من الأمثلة القصيرة لحالات استخدام مختلفة. 


## جدول المحتويات  

[نظرة عامة](#overview)  
[كيفية استخدام خدمة OpenAI](#how-to-use-openai-service)  
[1. إنشاء خدمة OpenAI الخاصة بك](#1.-creating-your-openai-service)  
[2. التثبيت](#2.-installation)    
[3. أوراق الاعتماد](#3.-credentials)  

[حالات الاستخدام](#use-cases)    
[1. تلخيص النص](#1.-summarize-text)  
[2. تصنيف النص](#2.-classify-text)  
[3. توليد أسماء منتجات جديدة](#3.-generate-new-product-names)  
[4. ضبط دقيق لمصنف](#4.fine-tune-a-classifier)  

[المراجع](#references)


### أنشئ طلبك الأول  
سيوفر هذا التمرين القصير مقدمة أساسية لإرسال الطلبات إلى نموذج OpenAI لمهمة بسيطة "التلخيص".  


**الخطوات**:  
1. قم بتثبيت مكتبة OpenAI في بيئة بايثون الخاصة بك  
2. قم بتحميل المكتبات المساعدة القياسية وقم بضبط بيانات الاعتماد الأمنية المعتادة الخاصة بـ OpenAI لخدمة OpenAI التي أنشأتها  
3. اختر نموذجًا لمهمتك  
4. أنشئ طلبًا بسيطًا للنموذج  
5. أرسل طلبك إلى واجهة برمجة التطبيقات للنموذج!  


### 1. تثبيت OpenAI


In [ ]:
%pip install openai python-dotenv

### ٢. استيراد مكتبات المساعدة وإنشاء بيانات الاعتماد


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. إيجاد النموذج المناسب  
النماذج مثل GPT-4o و GPT-4o mini يمكنها فهم وتوليد اللغة الطبيعية، ومتاحة على منصة OpenAI بمستويات مختلفة من القوة والسرعة تناسب مهامًا مختلفة.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. تصميم الموجه  

"السحر في نماذج اللغة الكبيرة هو أنه من خلال تدريبها على تقليل خطأ التنبؤ هذا على كميات هائلة من النصوص، تنتهي النماذج بتعلم مفاهيم مفيدة لهذه التنبؤات. على سبيل المثال، فهي تتعلم مفاهيم مثل"(1):

* كيفية التهجئة
* كيفية عمل القواعد النحوية
* كيفية إعادة الصياغة
* كيفية الإجابة على الأسئلة
* كيفية إجراء محادثة
* كيفية الكتابة بعدة لغات
* كيفية البرمجة
* إلخ.

#### كيفية التحكم في نموذج لغة كبير  
"من بين جميع المدخلات إلى نموذج اللغة الكبير، فإن أكثرها تأثيرًا هو موجه النص(1).

يمكن تحفيز نماذج اللغة الكبيرة لإنتاج مخرجات بعدة طرق:

التعليمات: أخبر النموذج بما تريد
الإكمال: حث النموذج على استكمال بداية ما تريده
العرض: أرِ النموذج ما تريده، إما من خلال:
بعض الأمثلة في الموجه
مئات أو آلاف الأمثلة في مجموعة تدريب دقيقة"



#### هناك ثلاث إرشادات أساسية لإنشاء الموجهات:

**أظهر وقل**. اجعل ما تريد واضحًا إما عبر التعليمات، أو الأمثلة، أو مزيج من الاثنين. إذا كنت تريد من النموذج ترتيب قائمة من العناصر بترتيب أبجدي أو تصنيف فقرة حسب المشاعر، فأرِه هذا هو ما تريده.

**قدم بيانات ذات جودة**. إذا كنت تحاول بناء مصنف أو جعل النموذج يتبع نمطًا معينًا، فتأكد من وجود ما يكفي من الأمثلة. تأكد من تدقيق أمثلتك — النموذج عادة ما يكون ذكيًا بما يكفي لرؤية الأخطاء الإملائية الأساسية ويعطيك ردًا، لكنه قد يفترض أن هذا مقصود وقد يؤثر على الرد.

**تحقق من إعداداتك.** تحكم إعدادات درجة الحرارة وtop_p إلى أي مدى يكون النموذج حتمياً في توليد الرد. إذا كنت تطلب ردًا حيث يوجد جواب صحيح واحد فقط، سترغب في ضبط هذه الإعدادات أقل. إذا كنت تبحث عن ردود أكثر تنوعًا، فقد ترغب في رفعها. الخطأ الأول الذي يرتكبه الناس مع هذه الإعدادات هو افتراض أنها تحكم "الذكاء" أو "الإبداع".


المصدر: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. الإرسال!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### كرر المكالمة نفسها، كيف تقارن النتائج؟  


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## تلخيص النص  
#### التحدي  
تلخيص النص بإضافة "tl;dr:" في نهاية فقرة النص. لاحظ كيف يفهم النموذج كيفية القيام بعدد من المهام بدون تعليمات إضافية. يمكنك تجربة مطالبات أكثر وصفًا من tl;dr لتعديل سلوك النموذج وتخصيص التلخيص الذي تحصل عليه(3).  

أظهرت الأبحاث الحديثة تحقيق مكاسب كبيرة في العديد من مهام ومعايير معالجة اللغة الطبيعية عبر التدريب المسبق على مجموعة كبيرة من النصوص تليه تحسينات دقيقة على مهمة محددة. ورغم أن هذا الأسلوب عادة ما يكون معماريًا غير مرتبط بالمهمة، إلا أنه لا يزال يتطلب مجموعات بيانات دقيقة خاصة بالمهمة تحتوي على آلاف أو عشرات الآلاف من الأمثلة. بالمقابل، يمكن للبشر عمومًا أداء مهمة جديدة في اللغة من خلال عدد قليل من الأمثلة أو من تعليمات بسيطة - وهو أمر لا تزال أنظمة معالجة اللغة الطبيعية الحالية تكافح لتحقيقه بشكل كبير. هنا نُظهر أن توسيع نماذج اللغة يحسن بشكل كبير الأداء غير المعتمد على المهمة في حالات التدرب القليل، وأحيانًا يصل حتى لمنافسة أفضل الطرق السابقة التي تعتمد على تحسينات دقيقة.  



ملخص  


# تمارين لعدة حالات استخدام  
1. تلخيص النص  
2. تصنيف النص  
3. توليد أسماء منتجات جديدة


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## تصنيف النص  
#### التحدي  
صنّف العناصر إلى فئات تُقدّم أثناء وقت الاستدلال. في المثال التالي، نوفر كلا من الفئات والنص ليتم تصنيفه في المطالَب (*playground_reference). 

استفسار العميل: مرحباً، أحد مفاتيح لوحة مفاتيح الكمبيوتر المحمول انكسر مؤخراً وأحتاج إلى بديل:

الفئة المصنفة:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## توليد أسماء منتجات جديدة
#### التحدي
إنشاء أسماء منتجات من كلمات أمثلة. هنا ندرج في الموجه معلومات عن المنتج الذي سنولد أسماء له. كما نُقدم مثالًا مشابهًا لعرض النمط الذي نرغب في الحصول عليه. لقد قمنا أيضًا بضبط قيمة درجة الحرارة عالية لزيادة العشوائية والحصول على استجابات أكثر ابتكارًا.

وصف المنتج: صانع مخفوق الحليب المنزلي
كلمات الأساس: سريع، صحي، مضغوط.
أسماء المنتجات: HomeShaker, Fit Shaker, QuickShake, Shake Maker

وصف المنتج: زوج من الأحذية التي تناسب أي مقاس قدم.
كلمات الأساس: قابل للتكيف، مناسب، تناسب شامل.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# المراجع  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [بوابة Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [أفضل الممارسات لتحسين نماذج GPT لتصنيف النصوص](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# للمزيد من المساعدة  
[فريق تسويق OpenAI](AzureOpenAITeam@microsoft.com) 


# المساهمون
* لويس لي  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
